In [1]:
from pathlib import Path
import gc
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

# 프로젝트 루트
PROJECT_ROOT = Path.cwd()

# notebooks/에서 실행했다면 상위 폴더를 프로젝트 루트로 사용
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data/processed/network_ml_dev_v3/model_sample"
FEATURE_DIR = PROJECT_ROOT / "results/feature_selection"
MODEL_DIR = PROJECT_ROOT / "models/network"
RESULT_DIR = PROJECT_ROOT / "results/model_diversity_v3"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = DATA_DIR / "train_v3_sample.parquet"
VALID_PATH = DATA_DIR / "valid_v3_sample.parquet"

TOP40_PATH = FEATURE_DIR / "top40_features.csv"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("TRAIN        :", TRAIN_PATH)
print("VALID        :", VALID_PATH)
print("TOP40        :", TOP40_PATH)
print("RESULT       :", RESULT_DIR)

PROJECT_ROOT : /home/geonug/kdt-linux/ML
TRAIN        : /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v3/model_sample/train_v3_sample.parquet
VALID        : /home/geonug/kdt-linux/ML/data/processed/network_ml_dev_v3/model_sample/valid_v3_sample.parquet
TOP40        : /home/geonug/kdt-linux/ML/results/feature_selection/top40_features.csv
RESULT       : /home/geonug/kdt-linux/ML/results/model_diversity_v3


In [2]:
required_files = [
    TRAIN_PATH,
    VALID_PATH,
    TOP40_PATH,
]

for path in required_files:
    print(f"{path.name:<30} exists={path.exists()}")

assert TRAIN_PATH.exists(), TRAIN_PATH
assert VALID_PATH.exists(), VALID_PATH
assert TOP40_PATH.exists(), TOP40_PATH

print("\n환경 확인 완료")

train_v3_sample.parquet        exists=True
valid_v3_sample.parquet        exists=True
top40_features.csv             exists=True

환경 확인 완료


In [3]:
top40_df = pd.read_csv(TOP40_PATH)

print(top40_df.head())
print("\ncolumns:", top40_df.columns.tolist())
print("shape:", top40_df.shape)

   rank            feature
0     1  Fwd Act Data Pkts
1     2   Fwd Seg Size Min
2     3     Fwd Header Len
3     4       ECE Flag Cnt
4     5  Init Fwd Win Byts

columns: ['rank', 'feature']
shape: (40, 2)


In [4]:
possible_columns = [
    "feature",
    "Feature",
    "feature_name",
    "Feature Name",
]

feature_col = None

for col in possible_columns:
    if col in top40_df.columns:
        feature_col = col
        break

if feature_col is None:
    # 첫 번째 문자열 컬럼 fallback
    object_cols = top40_df.select_dtypes(include="object").columns.tolist()

    if not object_cols:
        raise ValueError(
            f"Feature 이름 컬럼을 찾지 못했습니다. columns={top40_df.columns.tolist()}"
        )

    feature_col = object_cols[0]

TOP40_FEATURES = top40_df[feature_col].astype(str).tolist()

print("Feature column :", feature_col)
print("Feature count  :", len(TOP40_FEATURES))
print()

for i, feature in enumerate(TOP40_FEATURES, 1):
    print(f"{i:02d}. {feature}")

assert len(TOP40_FEATURES) == 40

Feature column : feature
Feature count  : 40

01. Fwd Act Data Pkts
02. Fwd Seg Size Min
03. Fwd Header Len
04. ECE Flag Cnt
05. Init Fwd Win Byts
06. URG Flag Cnt
07. RST Flag Cnt
08. Dst Port
09. Fwd PSH Flags
10. Fwd Pkts/s
11. Fwd Pkt Len Mean
12. Pkt Len Max
13. SYN Flag Cnt
14. Fwd Seg Size Avg
15. Bwd Header Len
16. Init Bwd Win Byts
17. Fwd IAT Std
18. Fwd IAT Max
19. Pkt Size Avg
20. Fwd IAT Tot
21. Bwd Pkt Len Max
22. PSH Flag Cnt
23. Flow IAT Mean
24. Bwd IAT Min
25. Bwd Pkt Len Std
26. Pkt Len Mean
27. Bwd IAT Max
28. Fwd Pkt Len Std
29. ACK Flag Cnt
30. TotLen Bwd Pkts
31. Pkt Len Std
32. FIN Flag Cnt
33. Bwd IAT Mean
34. Flow Duration
35. Flow IAT Max
36. Subflow Bwd Byts
37. Idle Max
38. Fwd URG Flags
39. Bwd Pkts/s
40. Bwd IAT Std


In [5]:
LOAD_COLUMNS = TOP40_FEATURES + ["label_id"]

print("Loading Train...")
train_df = pd.read_parquet(
    TRAIN_PATH,
    columns=LOAD_COLUMNS,
)

print("Loading Validation...")
valid_df = pd.read_parquet(
    VALID_PATH,
    columns=LOAD_COLUMNS,
)

print("\nTrain:", train_df.shape)
print("Valid:", valid_df.shape)

print(
    "Approx memory:",
    round(
        (
            train_df.memory_usage(deep=True).sum()
            + valid_df.memory_usage(deep=True).sum()
        )
        / 1024**3,
        3,
    ),
    "GB",
)

Loading Train...
Loading Validation...

Train: (702217, 41)
Valid: (142082, 41)
Approx memory: 0.253 GB


In [6]:
X_train = train_df[TOP40_FEATURES].to_numpy(dtype=np.float32)
y_train = train_df["label_id"].to_numpy(dtype=np.int16)

X_valid = valid_df[TOP40_FEATURES].to_numpy(dtype=np.float32)
y_valid = valid_df["label_id"].to_numpy(dtype=np.int16)

del train_df
del valid_df

gc.collect()

print("X_train:", X_train.shape, X_train.dtype)
print("X_valid:", X_valid.shape, X_valid.dtype)
print("y_train:", y_train.shape, y_train.dtype)
print("y_valid:", y_valid.shape, y_valid.dtype)

print(
    "\nArray memory:",
    round(
        (
            X_train.nbytes
            + X_valid.nbytes
            + y_train.nbytes
            + y_valid.nbytes
        )
        / 1024**3,
        3,
    ),
    "GB",
)

X_train: (702217, 40) float32
X_valid: (142082, 40) float32
y_train: (702217,) int16
y_valid: (142082,) int16

Array memory: 0.127 GB


In [7]:
print("Train NaN:", np.isnan(X_train).sum())
print("Train Inf:", np.isinf(X_train).sum())

print("Valid NaN:", np.isnan(X_valid).sum())
print("Valid Inf:", np.isinf(X_valid).sum())

Train NaN: 0
Train Inf: 0
Valid NaN: 0
Valid Inf: 0


In [8]:
# Inf → NaN
X_train[~np.isfinite(X_train)] = np.nan
X_valid[~np.isfinite(X_valid)] = np.nan

# Train 데이터에서만 median 계산
train_medians = np.nanmedian(X_train, axis=0).astype(np.float32)

# 전부 NaN인 feature 방어
train_medians = np.where(
    np.isfinite(train_medians),
    train_medians,
    0.0,
).astype(np.float32)

print("Median calculated:", train_medians.shape)

Median calculated: (40,)


In [9]:
for col_idx in range(X_train.shape[1]):

    train_mask = np.isnan(X_train[:, col_idx])

    if train_mask.any():
        X_train[train_mask, col_idx] = train_medians[col_idx]

    valid_mask = np.isnan(X_valid[:, col_idx])

    if valid_mask.any():
        X_valid[valid_mask, col_idx] = train_medians[col_idx]

print("After imputation")

print("Train finite:", np.isfinite(X_train).all())
print("Valid finite:", np.isfinite(X_valid).all())

After imputation
Train finite: True
Valid finite: True


In [10]:
SCALER_PATH = MODEL_DIR / "standard_scaler_top40_v3.joblib"

scaler = StandardScaler(copy=False)

start = time.perf_counter()

X_train = scaler.fit_transform(X_train)
X_valid = scaler.transform(X_valid)

scaling_time = time.perf_counter() - start

# 다시 float32 유지
X_train = X_train.astype(np.float32, copy=False)
X_valid = X_valid.astype(np.float32, copy=False)

joblib.dump(scaler, SCALER_PATH)

print(f"Scaling time : {scaling_time:.2f}s")
print("Scaler saved :", SCALER_PATH)
print("Train dtype  :", X_train.dtype)
print("Valid dtype  :", X_valid.dtype)

Scaling time : 0.66s
Scaler saved : /home/geonug/kdt-linux/ML/models/network/standard_scaler_top40_v3.joblib
Train dtype  : float32
Valid dtype  : float32


In [11]:
def evaluate_model(
    model_name,
    model,
    X,
    y,
    result_dir,
):
    result_dir = Path(result_dir)
    result_dir.mkdir(parents=True, exist_ok=True)

    start = time.perf_counter()

    pred = model.predict(X)

    inference_time = time.perf_counter() - start

    accuracy = accuracy_score(y, pred)

    macro_precision, macro_recall, macro_f1, _ = (
        precision_recall_fscore_support(
            y,
            pred,
            average="macro",
            zero_division=0,
        )
    )

    _, _, weighted_f1, _ = precision_recall_fscore_support(
        y,
        pred,
        average="weighted",
        zero_division=0,
    )

    metrics = {
        "model": model_name,
        "accuracy": accuracy,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "validation_samples": len(y),
        "inference_seconds": inference_time,
    }

    pd.DataFrame([metrics]).to_csv(
        result_dir / "validation_metrics.csv",
        index=False,
    )

    report = classification_report(
        y,
        pred,
        output_dict=True,
        zero_division=0,
    )

    pd.DataFrame(report).T.to_csv(
        result_dir / "classification_report.csv"
    )

    cm = confusion_matrix(y, pred)

    pd.DataFrame(cm).to_csv(
        result_dir / "confusion_matrix.csv",
        index=False,
    )

    np.save(
        result_dir / "validation_predictions.npy",
        pred,
    )

    print(f"\n=== {model_name} ===")

    for key in [
        "accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "weighted_f1",
    ]:
        print(f"{key:<20}: {metrics[key]:.6f}")

    print(
        f"inference          : "
        f"{inference_time:.3f}s "
        f"({len(y) / inference_time:,.0f} samples/s)"
    )

    return metrics

In [12]:
LOGISTIC_MODEL_PATH = (
    MODEL_DIR / "logistic_regression_top40_v3.joblib"
)

LOGISTIC_RESULT_DIR = (
    PROJECT_ROOT / "results/logistic_regression_top40_v3"
)

logistic_model = LogisticRegression(
    solver="saga",
    penalty="l2",
    C=1.0,
    class_weight="balanced",
    max_iter=300,
    random_state=42,
    n_jobs=2,
    verbose=1,
)

print("Training Logistic Regression...")

start = time.perf_counter()

logistic_model.fit(
    X_train,
    y_train,
)

train_time = time.perf_counter() - start

print(f"\nTraining time: {train_time / 60:.2f} min")

Training Logistic Regression...
Epoch 1, change: 1
Epoch 2, change: 0.28878951
Epoch 3, change: 0.1497547
Epoch 4, change: 0.10358328
Epoch 5, change: 0.081707232
Epoch 6, change: 0.057942707
Epoch 7, change: 0.080710031
Epoch 8, change: 0.063429654
Epoch 9, change: 0.10603989
Epoch 10, change: 0.044429485
Epoch 11, change: 0.045718815
Epoch 12, change: 0.040777896
Epoch 13, change: 0.042881787
Epoch 14, change: 0.094140038
Epoch 15, change: 0.020163685
Epoch 16, change: 0.023037966
Epoch 17, change: 0.019885378
Epoch 18, change: 0.11207914
Epoch 19, change: 0.052378587
Epoch 20, change: 0.023684299
Epoch 21, change: 0.023124877
Epoch 22, change: 0.020519387
Epoch 23, change: 0.021334445
Epoch 24, change: 0.029845577
Epoch 25, change: 0.031320017
Epoch 26, change: 0.016811468
Epoch 27, change: 0.036117017
Epoch 28, change: 0.043863211
Epoch 29, change: 0.045470055
Epoch 30, change: 0.017396653
Epoch 31, change: 0.048273921
Epoch 32, change: 0.11009213
Epoch 33, change: 0.1170587
Epoch 

In [13]:
joblib.dump(
    logistic_model,
    LOGISTIC_MODEL_PATH,
    compress=3,
)

print("Saved:", LOGISTIC_MODEL_PATH)
print(
    "Size:",
    round(LOGISTIC_MODEL_PATH.stat().st_size / 1024**2, 2),
    "MB",
)

Saved: /home/geonug/kdt-linux/ML/models/network/logistic_regression_top40_v3.joblib
Size: 0.0 MB


In [14]:
logistic_metrics = evaluate_model(
    model_name="LogisticRegression-Top40",
    model=logistic_model,
    X=X_valid,
    y=y_valid,
    result_dir=LOGISTIC_RESULT_DIR,
)

logistic_metrics["train_seconds"] = train_time

pd.DataFrame([logistic_metrics]).to_csv(
    LOGISTIC_RESULT_DIR / "validation_metrics.csv",
    index=False,
)


=== LogisticRegression-Top40 ===
accuracy            : 0.831273
macro_precision     : 0.576163
macro_recall        : 0.769018
macro_f1            : 0.588358
weighted_f1         : 0.834737
inference          : 0.082s (1,734,745 samples/s)


In [15]:
del logistic_model

gc.collect()

print("Logistic Regression released.")

Logistic Regression released.


In [16]:
MLP_MODEL_PATH = (
    MODEL_DIR / "mlp_top40_v3.joblib"
)

MLP_RESULT_DIR = (
    PROJECT_ROOT / "results/mlp_top40_v3"
)

mlp_model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",

    batch_size=2048,

    learning_rate="adaptive",
    learning_rate_init=0.001,

    max_iter=100,

    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=8,

    random_state=42,
    verbose=True,
)

print("Training MLP...")

start = time.perf_counter()

mlp_model.fit(
    X_train,
    y_train,
)

train_time = time.perf_counter() - start

print(f"\nTraining time: {train_time / 60:.2f} min")
print("Iterations   :", mlp_model.n_iter_)

Training MLP...
Iteration 1, loss = 0.69578188
Validation score: 0.887585
Iteration 2, loss = 0.22643638
Validation score: 0.901341
Iteration 3, loss = 0.19134303
Validation score: 0.905129
Iteration 4, loss = 0.17644753
Validation score: 0.910470
Iteration 5, loss = 0.16893678
Validation score: 0.912862
Iteration 6, loss = 0.16442408
Validation score: 0.914101
Iteration 7, loss = 0.16118548
Validation score: 0.914585
Iteration 8, loss = 0.15887805
Validation score: 0.914329
Iteration 9, loss = 0.15730857
Validation score: 0.915041
Iteration 10, loss = 0.15584829
Validation score: 0.917177
Iteration 11, loss = 0.15469474
Validation score: 0.914144
Iteration 12, loss = 0.15359347
Validation score: 0.917547
Iteration 13, loss = 0.15277676
Validation score: 0.916593
Iteration 14, loss = 0.15197285
Validation score: 0.917205
Iteration 15, loss = 0.15152075
Validation score: 0.917590
Iteration 16, loss = 0.15106381
Validation score: 0.914770
Iteration 17, loss = 0.15033168
Validation score:

In [17]:
joblib.dump(
    mlp_model,
    MLP_MODEL_PATH,
    compress=3,
)

print("Saved:", MLP_MODEL_PATH)

print(
    "Size:",
    round(MLP_MODEL_PATH.stat().st_size / 1024**2, 2),
    "MB",
)

Saved: /home/geonug/kdt-linux/ML/models/network/mlp_top40_v3.joblib
Size: 0.06 MB


In [18]:
mlp_metrics = evaluate_model(
    model_name="MLP-Top40",
    model=mlp_model,
    X=X_valid,
    y=y_valid,
    result_dir=MLP_RESULT_DIR,
)

mlp_metrics["train_seconds"] = train_time

pd.DataFrame([mlp_metrics]).to_csv(
    MLP_RESULT_DIR / "validation_metrics.csv",
    index=False,
)


=== MLP-Top40 ===
accuracy            : 0.923495
macro_precision     : 0.881728
macro_recall        : 0.809272
macro_f1            : 0.827712
weighted_f1         : 0.923395
inference          : 0.113s (1,257,176 samples/s)


In [19]:
del mlp_model

gc.collect()

print("MLP released.")

MLP released.


In [20]:
comparison_rows = []

result_sources = {
    "XGBoost-Top40":
        PROJECT_ROOT / "results/xgboost_top40_v3/validation_metrics.csv",

    "RandomForest-Top40":
        PROJECT_ROOT / "results/random_forest_top40_v3/validation_metrics.csv",

    "HistGradientBoosting-Top40":
        PROJECT_ROOT / "results/hist_gradient_boosting_top40_v3/validation_metrics.csv",

    "LightGBM-Top40":
        PROJECT_ROOT / "results/lightgbm_top40_v3/validation_metrics.csv",

    "LogisticRegression-Top40":
        PROJECT_ROOT / "results/logistic_regression_top40_v3/validation_metrics.csv",

    "MLP-Top40":
        PROJECT_ROOT / "results/mlp_top40_v3/validation_metrics.csv",
}

for model_name, path in result_sources.items():

    if not path.exists():
        print("Missing:", path)
        continue

    df = pd.read_csv(path)

    row = df.iloc[0].to_dict()
    row["model"] = model_name

    comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)

columns = [
    "model",
    "accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "weighted_f1",
]

comparison_df = comparison_df[
    [c for c in columns if c in comparison_df.columns]
]

comparison_df = comparison_df.sort_values(
    "macro_f1",
    ascending=False,
)

print(
    comparison_df.to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

                     model  accuracy  macro_precision  macro_recall  macro_f1  weighted_f1
            LightGBM-Top40  0.945792         0.871111      0.885821  0.876563     0.945745
             XGBoost-Top40  0.943117         0.859209      0.863805  0.859573     0.943070
HistGradientBoosting-Top40  0.941294         0.830949      0.867912  0.844567     0.941231
                 MLP-Top40  0.923495         0.881728      0.809272  0.827712     0.923395
        RandomForest-Top40  0.942991         0.806374      0.852425  0.825171     0.942943
  LogisticRegression-Top40  0.831273         0.576163      0.769018  0.588358     0.834737


In [21]:
FINAL_COMPARISON_DIR = (
    PROJECT_ROOT / "results/model_comparison_v3"
)

FINAL_COMPARISON_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

comparison_path = (
    FINAL_COMPARISON_DIR
    / "model_diversity_comparison.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

print("Saved:", comparison_path)

Saved: /home/geonug/kdt-linux/ML/results/model_comparison_v3/model_diversity_comparison.csv


In [22]:
lgbm_row = comparison_df[
    comparison_df["model"] == "LightGBM-Top40"
].iloc[0]

comparison_delta = comparison_df.copy()

comparison_delta["macro_f1_vs_lgbm"] = (
    comparison_delta["macro_f1"]
    - lgbm_row["macro_f1"]
)

comparison_delta["accuracy_vs_lgbm"] = (
    comparison_delta["accuracy"]
    - lgbm_row["accuracy"]
)

print(
    comparison_delta[
        [
            "model",
            "accuracy",
            "macro_f1",
            "accuracy_vs_lgbm",
            "macro_f1_vs_lgbm",
        ]
    ].to_string(
        index=False,
        float_format=lambda x: f"{x:.6f}",
    )
)

                     model  accuracy  macro_f1  accuracy_vs_lgbm  macro_f1_vs_lgbm
            LightGBM-Top40  0.945792  0.876563          0.000000          0.000000
             XGBoost-Top40  0.943117  0.859573         -0.002675         -0.016990
HistGradientBoosting-Top40  0.941294  0.844567         -0.004497         -0.031996
                 MLP-Top40  0.923495  0.827712         -0.022297         -0.048852
        RandomForest-Top40  0.942991  0.825171         -0.002801         -0.051392
  LogisticRegression-Top40  0.831273  0.588358         -0.114518         -0.288205
